## Economics-Driven Rule Agent (EcoBot v7) + Arena + Analytics

**A fully reactive agent, not a scripted playbook.** Sell orders are re-priced every single turn against the live market and shed contents; purchase decisions (land, herd, seeds) are recomputed fresh from the current `obs` at the start of every day rather than on fixed calendar triggers — there are no `if day == N: buy X` branches. The only hard-coded anchors are season boundaries (per-crop planting deadlines, day-28 liquidation) and a handful of tuned constants (cash reserve, shed capacity, hand-hire floor).

**Day 0 Opener.** Hour 0: arbitrage-buy 53 wheat and hire a floor of 4 hands. Hour 1 — gated to fire only while the shed still holds zero sheep and the field zero animals — liquidates that wheat down to a 4-unit reserve, buys the 2-cow/2-sheep starter herd, and requests starter seed targets (12 melon, 9 wheat), trimmed by whatever cash settlement allows that turn.

**Target-Based Macro Planner.** Each crop/animal target is `base allocation + per_shop increment × matching unlocked shop instances` (e.g. +3 strawberry, +4 sheep, +3 cows, +2 geese, +6 tomato per matching shop), capped at a hard per-asset ceiling. Wheat additionally gets bumped up to cover projected feed for the currently kept herd. Targets recompute at hours 0/1/2/PURCHASE_HOUR each day and are cached in between — an explicit `RuntimeError` fires if that cache is ever missing, rather than silently reusing stale data.

**Daily Field Dispatch — real VRP, not a priority queue.** Once per day (rebuilt through hour 2 as hired hands spawn), the full task list — feed, water, harvest, fertilize, care, place, build, weed — is solved by parallel cheapest insertion: each task lands on whichever unit's route absorbs it at the lowest travel-cost delta. An urgency ranking (starvation-risk feed > placing new animals > critical watering > harvest > fertilize > routine upkeep > planting > weed/expired cleanup) only decides evaluation order, never overrides distance. `PLANT+WATER` and `BUILD+PLACE` are bundled onto one stop so no task is left half-done, and a binary search (`required_hand_count`) picks the minimum headcount (up to 13) that clears the day's backlog.

**Capacity-Driven Land Expansion.** Quadrants unlock in a fixed order (NE → SW) purely when tile utilization ≥85% and either crop demand or animal-structure needs exceed current capacity, and the budget covers the cost plus reserve. Under the current tuning (`MAX_UNLOCKED_QUADS = 3`) the SE quadrant is never purchased — the farm caps out at 3 of 4 quadrants by design.

**Crop & Season Phasing.** Planting windows fall out of each crop's yield lag versus the 30-day season: melon and strawberry close around day 19, tomato around day 21, wheat/carrot stay open through days 25–26. Past strawberry's cutoff, leftover seed budget switches entirely to whichever staple (wheat vs. carrot) currently nets the higher spot profit per tile. Day 28 starts full liquidation of everything sellable; day 29 drops the feed-floor requirement entirely.

**Cash & Shed Risk Management.** A $140 cash reserve is enforced on days 1–27 for land/hire orders (seed and product buys may spend into it so a freshly unlocked quadrant gets planted immediately). If projected shed occupancy hits 60% of capacity, every reserve floor is bypassed and the bot dumps at any price; a separate field-level guard caps `DROP`/`PLACE` actions at remaining shed space so cargo is never silently discarded — overflow is logged loudly instead.

**Order-Cap Prioritization.** With only 10 market-order slots per turn, sells get priority room, then the mandatory hire floor, then the rest of the plan, then any overflow sells and extra hires. Orders dropped by the cap are rolled back out of the seasonal seed-purchase ledger so future turns don't undercount remaining budget.

In [ ]:
!pip install -q -U kaggle-environments

## 1. Setup: Load Project Files from Dataset
Copies bot sources, arena runner, and analytics script from the attached dataset (`/kaggle/input/kaggricult-eco7lite`) into `/kaggle/working`.

In [ ]:
import shutil
from pathlib import Path

# Automatically find the dataset directory by locating arena.py
dataset_dir = next(Path("/kaggle/input").rglob("arena.py")).parent
print(f"Found dataset at: {dataset_dir}")

# Copy bots folder and evaluation scripts into /kaggle/working
shutil.copytree(dataset_dir / "bots", "bots", dirs_exist_ok=True)
for py_file in dataset_dir.glob("*.py"):
    shutil.copy2(py_file, py_file.name)

print("Files successfully copied:")
!ls -la


## 2. Build Submission: Standalone Single-File Bundle
Packages the modular `eco7Lite` codebase into a single standalone `submission.py` via `bundle.py`.

In [ ]:
!python bots/eco7Lite/bundle.py
!mv bots/eco7Lite/submission.py ./submission.py

## 3. Verify Submission Structure (AST Check)
Verifies that `submission.py` parses cleanly without syntax errors and exposes the mandatory top-level `agent` function.

In [ ]:
import ast

with open("submission.py") as f:
    tree = ast.parse(f.read())

has_agent = any(isinstance(n, ast.FunctionDef) and n.name == "agent" for n in tree.body)
assert has_agent, "submission.py must define a top-level `agent` function"
print("OK: submission.py defines `agent()` and parses cleanly without syntax errors.")


## 4. Quick Sanity Check
A short 48-turn run (2 in-game days) against `"random"` to catch obvious runtime issues before full matches.

In [ ]:
from kaggle_environments import make

env = make("kaggriculture", configuration={"episodeSteps": 48}, debug=True)
env.run(["submission.py", "random"])

final = env.steps[-1]
for i, s in enumerate(final):
    print(f"Player {i}: reward={s.reward}, status={s.status}")


## 5. Multi-Game Evaluation (Arena Benchmark)
Parallel evaluation across multiple random seeds with symmetric pairings ($P_0$ vs $P_1$ and $P_1$ vs $P_0$) benchmarking `submission.py` against the `starter` baseline.

In [ ]:
!python arena.py --bots submission.py starter --games 2 --workers 4 --seed 42 --save-replays both

## 6. Financial & P&L Statement Analytics
Audits unit economics and sector-by-sector profit margins (commercial crops, grain/feed, livestock, labor, land) across the arena match replays.

In [ ]:
!python bot_pnl.py "replays/*.json" --bot submission

## 7. Cleanup Temporary Files
Cleans up Python byte-code caches to ensure a minimal, clean submission output directory.

In [ ]:
!find /kaggle/working -type d -name "__pycache__" -exec rm -rf {} +
!find /kaggle/working -type f -name "*.pyc" -delete